# Structured Output 
Models can be reusted to prvide theri respnse in  a format mathcing a given shcema. This is useful for ensuring the output can be easily parsed and used in subsequent procesing. LangChain supports multiple schema types and methods for enforcing structured output 

# Pydantic 
Pydantic models provide the riches set with field validation , descriptions and nested structures

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]= os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-3.5-flash")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12', 'langchain-google-genai': '4.2.7'}}, output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.5-flash', temperature=1.0, client=<google.genai.client.Client object at 0x00000201C30F04A0>, default_metadata=(), model_kwargs={})

In [3]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="Title of the Movie")
    year:int=Field(description="This year of the movie was relesed")
    director:str =Field(description="The director of the movie")
    rating:float= Field(description="The movies rating out of 10")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12', 'langchain-google-genai': '4.2.7'}}, output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.5-flash', temperature=1.0, client=<google.genai.client.Client object at 0x00000201C30F04A0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/jso

In [5]:
model.invoke("Provide deatials about the movie Inception")


AIMessage(content=[{'type': 'text', 'text': '**Inception** (2010) is a critically acclaimed science-fiction action heist film written and directed by Christopher Nolan. Known for its complex, multi-layered plot, stunning visual effects, and philosophical themes, it is widely considered one of the best films of the 21st century.\n\nHere is a detailed breakdown of the movie, covering its plot, cast, rules of the world, themes, production, and its famous ending.\n\n---\n\n### 1. Fast Facts\n*   **Release Date:** July 16, 2010\n*   **Director & Writer:** Christopher Nolan\n*   **Budget:** $160 million\n*   **Box Office:** $839 million worldwide\n*   **Accolades:** Nominated for 8 Academy Awards (including Best Picture); won 4 (Best Cinematography, Best Sound Editing, Best Sound Mixing, and Best Visual Effects).\n\n---\n\n### 2. The Premise\nThe film is set in a world where technology exists to enter people’s minds through their dreams (called "shared dreaming"). This technology is primaril

In [6]:
response= model_with_structure.invoke("Provide detials about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [7]:
type(response)

__main__.Movie

# Message output alongside parsed strucutre


In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,descripton="The year movie was released")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(...,description="The movies ratigng out of 10")

model_with_structure= model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")

response

C:\Users\allis\AppData\Local\Temp\ipykernel_10844\1037985890.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descripton'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year: int = Field(...,descripton="The year movie was released")


{'raw': AIMessage(content=[{'type': 'text', 'text': '{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}', 'extras': {'signature': 'EpgJCpUJARFNMg+RUl4bXhnA0Yxmo+Suj7SgozlkBBsZu393G2ExT3nMLj0dbQycYzReQhNiCAXQtfyG44cuGQaxii+q8gR3QSwtrz+pqGZ5Y82j5rUmSE/lBLufopYxMX3NXB+BBuf8JvOlkjsoLU5pPQAnqLWm8JuSgOtBcokBvwqPYw9Ur8IMq8Akm0ABu3zims9P5mu/Ln2KN4pxfqQ7wYTH8SvC7v0xNij9aK4rj+E48pI++zcT/SyyPGZVEFaovpXrh+707781HWvm203ubrSOZdAr0A4lkP2E02QZ4kMG2IFqE0Jg3dSICurB0SE7MC76SCvNXR65iYStZ9lKGLmwjll2iYKDU69Kfwmh5eu+whAweiZALF+DYSabhObJmYdubd4OjUBWCs6N0x+8dWxJNrv+4lFj7X4GoFfmiEA7pCe+TbHm8vKRoZyfP8+nTo5Tn0Ufox+gas053rfOf5/P9pkzax2MQYFFP3Fl1Pvp0O7JWau+FFDlFW5JyV6zQdz3BK4ikjy1Fb3l2IA+2ZBvIOWQ+uxQFvx5H4U0KcVuvMzeEyKnqCQWsdkUBVOrlSMyWGDVdkDJ9QzA/+obNtqzqvldP1DjHxBjR5unouW2n00Zbg2yZnhcrCHYoby/aFJIMZqKS5cNjarJ8mLUdGeWlKe9xD+I9/BQp2cYEjlXet/wFStMUYKDq1Ps6XvtiB8cMBJGX6JkPwBiPOOLbcwjIf7amkmDKy6qQDzOLlpLmQ6b9aNiurHe8vn5aAuEEkK4GgUufWTmVfhgW+gIzQWB1gvCy319VKsZJ5jpDtk0zBPtgglAKc3AMZB/7En

# Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Sci-Fi', 'Adventure', 'Thriller'], budget=160.0)

# Typed Dict
TypeDict provides a simpler alternative using Python Built-in-typing , ideal when you don't need runtime validation

In [11]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"The year the movie was released"]
    director: Annotated[str,...,"Teh director of the movie"]
    rating: Annotated[float,...,"The movie rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response= model_withtypedict.invoke("PLease provide the details of the movie avengers")
response


{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [12]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
response =model_with_structure.invoke("Provide detials about the movie Avengers")
response

{'title': 'The Avengers',
 'year': '2012',
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'}],
 'genres': ['Action', 'Sci-Fi', 'Adventure'],
 'budget': 220000000}

In [13]:
model.profile

{'name': 'Gemini 3.5 Flash',
 'release_date': '2026-05-19',
 'last_updated': '2026-05-19',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

# Dataclasses

A data class is a class typically contaiing mainly data, although there are not really any restrictions. you create it using @dataclass decorator


In [ ]:
import os 
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


In [16]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='67f70594-549b-4855-99ff-4627937d6e0d'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 421, 'prompt_tokens': 204, 'total_tokens': 625, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E32nPsZRanlO5xHKvWBioIK7ruksF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f7632-5bd2-7b12-8d0c-8de0f0bc27ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={

In [17]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [18]:

## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [19]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')